In [1]:
import requests
import json
import pandas as pd
from bs4 import BeautifulSoup
import psycopg2, pymongo, elasticsearch


In [5]:
import requests
import json
import time
from dotenv import load_dotenv
import os

# Load API key from .env
load_dotenv()
API_KEY = os.getenv("API_KEY")  # Store your key in .env securely

SYMBOLS = ["AAPL", "MSFT", "GOOG", "AMZN", "META"]

# Function to fetch full historical data for a given symbol
def fetch_full_history(symbol):
    url = f"https://financialmodelingprep.com/api/v3/historical-price-full/{symbol}?apikey={API_KEY}"
    response = requests.get(url)
    if response.status_code == 200:  # ✅ Correct answer is 200
        data = response.json()
        return data.get("historical", [])  # Safely extract the history
    else:
        print(f"Error {response.status_code} for {symbol}")
    return []

# Dictionary to store all stock histories
stock_history = {}

# Fetch each symbol's history
for i, symbol in enumerate(SYMBOLS):
    print(f"Fetching full history for {symbol} ({i+1}/{len(SYMBOLS)})...")
    history = fetch_full_history(symbol)
    stock_history[symbol] = history
    time.sleep(1)  # Gentle delay to avoid rate limiting

# Save to JSON file
with open("full_stock_history.json", "w") as f:
    json.dump(stock_history, f, indent=2)

print("\n✅ Full historical data saved to 'full_stock_history.json'")




Fetching full history for AAPL (1/5)...
Fetching full history for MSFT (2/5)...
Fetching full history for GOOG (3/5)...
Fetching full history for AMZN (4/5)...
Fetching full history for META (5/5)...

✅ Full historical data saved to 'full_stock_history.json'


In [9]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

SYMBOLS = ["AAPL", "MSFT", "GOOG", "AMZN", "META"]

def scrape_finviz_info(symbol):
    url = f"https://finviz.com/quote.ashx?t={symbol}"
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    try:
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, "html.parser")

        # These tags worked for your success output
        sector = soup.find_all("a", class_="tab-link")[1]
        industry = soup.find_all("a", class_="tab-link truncate")[0]

        return {
            "symbol": symbol,
            "sector": sector.text.strip(),
            "industry": industry.text.strip()
        }

    except Exception as e:
        return {
            "symbol": symbol,
            "sector": "N/A",
            "industry": "N/A"
        }

company_profiles = [scrape_finviz_info(s) for s in SYMBOLS]
df = pd.DataFrame(company_profiles)
print(df)



  symbol                  sector                        industry
0   AAPL              Technology            Consumer Electronics
1   MSFT              Technology       Software - Infrastructure
2   GOOG  Communication Services  Internet Content & Information
3   AMZN       Consumer Cyclical                 Internet Retail
4   META  Communication Services  Internet Content & Information


### connection

In [10]:
import json
import psycopg2
import os
from dotenv import load_dotenv

# Load .env variables
load_dotenv()

# Load JSON data
with open("full_stock_history.json", "r", encoding="utf-8") as f:
    data = json.load(f)

symbols = data.keys()

# Use environment variables for DB credentials
conn = psycopg2.connect(
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT")
)

cur = conn.cursor()

# Insert stock data
for symbol in symbols:
    for record in data[symbol]:
        cur.execute("""
            INSERT INTO stock_quotes (
                symbol, date, open, high, low, close, adj_close,
                volume, unadjusted_volume, change, change_percent,
                vwap, label, change_over_time
            ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            symbol,
            record.get("date"),
            record.get("open"),
            record.get("high"),
            record.get("low"),
            record.get("close"),
            record.get("adjClose"),
            record.get("volume"),
            record.get("unadjustedVolume"),
            record.get("change"),
            record.get("changePercent"),
            record.get("vwap"),
            record.get("label"),
            record.get("changeOverTime")
        ))

conn.commit()
cur.close()
conn.close()


### MONGODB

In [14]:
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017/")
db = client["stocks"]
collection = db["profiles"]
collection.insert_many(company_profiles)

InsertManyResult([ObjectId('68404d7f0346eb85f1c83f3f'), ObjectId('68404d7f0346eb85f1c83f40'), ObjectId('68404d7f0346eb85f1c83f41'), ObjectId('68404d7f0346eb85f1c83f42'), ObjectId('68404d7f0346eb85f1c83f43')], acknowledged=True)

#### Testing MongoDb

In [15]:
import re

for doc in collection.find({"sector": re.compile("Technology")}):
    print(doc)
print("Total documents:", collection.count_documents({}))
tech_count = collection.count_documents({"sector": re.compile("Technology")})
print("Technology companies:", tech_count)


{'_id': ObjectId('68404d7f0346eb85f1c83f3f'), 'symbol': 'AAPL', 'sector': 'Technology', 'industry': 'Consumer Electronics'}
{'_id': ObjectId('68404d7f0346eb85f1c83f40'), 'symbol': 'MSFT', 'sector': 'Technology', 'industry': 'Software - Infrastructure'}
Total documents: 5
Technology companies: 2
